<a href="https://colab.research.google.com/github/Goseungeun/2026_BigData_Analyst/blob/main/Part4/Test_09.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Part1

In [6]:
# Q1
import pandas as pd
df = pd.read_csv('https://raw.githubusercontent.com/lovedlim/bae-v3/main/part4/ch9/loan.csv')

df['총대출액'] = df['신용대출'] + df['담보대출']

grouped = df.groupby(['지역코드','성별'])['총대출액'].sum().unstack()
grouped['diff'] = abs(grouped[1] - grouped[2])
ans = grouped['diff'].idxmax()

print(ans) # 4100000278

성별                 1         2      diff
지역코드                                    
4100000000  22891981  18339609   4552372
4100000001  34197683  17595557  16602126
4100000002  13119017  21827691   8708674
4100000003  16549091  54654134  38105043
4100000004  26831399  56145359  29313960
...              ...       ...       ...
4100000587   9050332  19445103  10394771
4100000588  47396699  36281031  11115668
4100000589  28256320  17797635  10458685
4100000590  16813363  23655928   6842565
4100000591  20674327  12204772   8469555

[592 rows x 3 columns]
4100000278


In [13]:
#Q2
import pandas as pd
df = pd.read_csv('https://raw.githubusercontent.com/lovedlim/bae-v3/main/part4/ch9/crime.csv')

cond1 = df['구분'] == '발생건수'
cond2 = df['구분'] == '검거건수'

df_1 = df[cond1].iloc[:,2:] # 발생건수
df_2 = df[cond2].iloc[:,2:] # 검거건수

df_1 = df_1.reset_index(drop=True)
df_2 = df_2.reset_index(drop=True)
df_3 = df_2/df_1

lll = df_3.idxmax(axis=1)

result = 0
for index,item in enumerate(lll):
  result = result + df_2.loc[index,item]

print(result)


7799


In [23]:
#Q3
import pandas as pd
df = pd.read_csv('https://raw.githubusercontent.com/lovedlim/bae-v3/main/part4/ch9/hr.csv')

df['만족도'] = df['만족도'].fillna(df['만족도'].mean())

gm = df.groupby(['부서','성과등급'])['근속연수'].transform('mean')
df['근속연수'] = df['근속연수'].fillna(gm)

df['peryear'] = df['연봉']/df['근속연수']
df_year = df.nlargest(3,'peryear')
ans1 = df_year.iloc[-1]['근속연수']

df['persati'] = df['연봉']/df['만족도']
df_sati = df.nlargest(2,'persati')
ans2 = df_sati.iloc[-1]['교육참가횟수']

result = ans1 + ans2
print(result)

7.0


## Part 2

In [31]:
import pandas as pd
train = pd.read_csv('https://raw.githubusercontent.com/lovedlim/bae-v3/main/part4/ch9/farm_train.csv')
test = pd.read_csv('https://raw.githubusercontent.com/lovedlim/bae-v3/main/part4/ch9/farm_test.csv')

#EDA
# print(train.shape)
# print(test.shape)

# print(train.info())
# print(train.isnull().sum().sum())
# print(train.isnull().sum().sum())

# print(train['지역'].nunique())
# print(test['지역'].nunique())

# print(train['작물종류'].nunique())
# print(test['작물종류'].nunique())

# print(train['토양유형'].nunique())
# print(test['토양유형'].nunique())

# print(train['등급'].nunique())
# print(test['등급'].nunique())

# Preprecessing
target = train.pop('농약검출여부')
train = pd.get_dummies(train)
test = pd.get_dummies(test)

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

X_train,X_val,y_train,y_val = train_test_split(train,target,test_size=0.2,random_state=0)
rf = RandomForestClassifier()
rf.fit(X_train,y_train)
val_pred = rf.predict(X_val)
print(f1_score(y_val,val_pred,average='macro'))

pred = rf.predict(test)
result = pd.DataFrame({'pred':pred})
result.to_csv('result.csv',index = False)

print(result.shape)
print(pd.read_csv('result.csv').head())

0.8631756009446812
(1000, 1)
   pred
0     2
1     0
2     0
3     2
4     0


## Part 3

In [37]:
#Q1
import pandas as pd
df = pd.read_csv('https://raw.githubusercontent.com/lovedlim/bae-v3/main/part4/ch9/design.csv')

train = df.iloc[:140]
test = df.iloc[140:]

# 1-1
from statsmodels.formula.api import ols
model = ols('design ~ c1 + c2 + c3 + c4',data=train).fit()
print(model.summary()) # 3개

# 1-2
model2 = ols('design ~ c1 + c2 + c4',data=train).fit()
train['pred_design'] = model2.predict(train)

print(round(train['pred_design'].corr(train['design']),3)) #0.501

# 1-3
test['pred'] = model2.predict(test)
test['diff'] = (test['pred'] - test['design'])**2

mse = test['diff'].mean()
print(mse**(1/2)) # 8.488

                            OLS Regression Results                            
Dep. Variable:                 design   R-squared:                       0.263
Model:                            OLS   Adj. R-squared:                  0.241
Method:                 Least Squares   F-statistic:                     12.03
Date:                Thu, 18 Jun 2026   Prob (F-statistic):           2.16e-08
Time:                        07:18:43   Log-Likelihood:                -469.00
No. Observations:                 140   AIC:                             948.0
Df Residuals:                     135   BIC:                             962.7
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     52.2556      2.034     25.695      0.0

/tmp/ipykernel_2461/2630672895.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train['pred_design'] = model2.predict(train)
/tmp/ipykernel_2461/2630672895.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test['pred'] = model2.predict(test)
/tmp/ipykernel_2461/2630672895.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/

In [44]:
# Q2
import pandas as pd
df = pd.read_csv('https://raw.githubusercontent.com/lovedlim/bae-v3/main/part4/ch9/retention.csv')

from statsmodels.formula.api import logit
model = logit('Churn~MonthlyCharges+CustomerTenure+HasPhoneService+HasTechInsurance',data=df).fit()
print(model.summary())
#2-1
print(round(model.pvalues['MonthlyCharges'],3)) # 0.008
#2-2
import numpy as np
print(round(np.exp(model.params['HasPhoneService']),3)) #0.701
#2-3
df['pred'] = model.predict(df)
cond1 = df['pred'] > 0.3
print(len(df[cond1])) #65

Optimization terminated successfully.
         Current function value: 0.582234
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:                  Churn   No. Observations:                   80
Model:                          Logit   Df Residuals:                       75
Method:                           MLE   Df Model:                            4
Date:                Thu, 18 Jun 2026   Pseudo R-squ.:                  0.1585
Time:                        07:27:26   Log-Likelihood:                -46.579
converged:                       True   LL-Null:                       -55.352
Covariance Type:            nonrobust   LLR p-value:                  0.001513
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept           -4.4731      1.437     -3.114      0.002      -7.289      -1.657
MonthlyChar